# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training,
)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor


In [2]:
from nba_ou.modeling.modeling import (
    TemporalDecaySampleWeightRegressor,
    assert_valid_time_splits,
    build_recency_sample_weights,
    evaluate_day_by_day_walk_forward,
    make_test_anchored_walk_forward_splits,
    save_model_bundle,
    load_model_bundle,
    split_latest_dates_holdout,
)


In [3]:
nan_threshold = 50.0
max_na_per_row = 70

## Load Data

In [4]:
exclude = "fanatics_sportsbook"

In [5]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260408.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [6]:
df_stats = df_stats[df_stats['SEASON_YEAR'] >= 2021]

In [7]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 6478 rows
Basic cleaning complete: 6463 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 2052 columns (896 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 1 (0.02%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 0/106
  Remaining NaN cells: 251178

Dropping rows with more than 70 NaN values...
Removed 663 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (5799, 2052)


In [8]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1730,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,761,13.12,2023.0
1728,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,748,12.90,2023.0
1734,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,678,11.69,2023.0
1732,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,667,11.50,2023.0
1729,total_consensus_pct_under_TREND_SLOPE_LAST_5_G...,654,11.28,2023.0
...,...,...,...,...
1884,spread_betmgm_price_home,1,0.02,2021.0
1873,total_betmgm_price_under,1,0.02,2021.0
1958,odds_ml_home_prob_novig_betmgm,1,0.02,2025.0
1959,odds_ml_vig_betmgm,1,0.02,2025.0


In [9]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [10]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [11]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1239
2022    1233
2023     967
2024    1238
2025    1122
dtype: int64


## Train / Test

In [12]:
TARGET_COL = "TOTAL_POINTS"
SAMPLE_WEIGHT_LAMBDA = 0.0075
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.015)
TRAIN_GAMES = 5100
DAY_BY_DAY_METRIC_NAME = "OU_Betting_Accuracy"
DAY_BY_DAY_THRESHOLDS = (1, 2, 3)


In [13]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 5501
Final test set size: 298
Final test date range: 2026-03-01 00:00:00 -> 2026-04-08 00:00:00


In [14]:
EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")
sample_weight_dev = build_recency_sample_weights(
    df_dev,
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(
    f"Recency sample weights lambda={SAMPLE_WEIGHT_LAMBDA}: "
    f"min={sample_weight_dev.min():.4f}, max={sample_weight_dev.max():.4f}"
)


X_dev shape: (5501, 2049)
X_test_final shape: (298, 2049)
Recency sample weights lambda=0.0075: min=0.0000, max=1.0000


In [15]:
from nba_ou.modeling.scorers import (
    OverUnderScorerTotalPoints,
    OverUnderScorerTotalPointsMinEdge,
    evaluate_total_points_thresholds,
    over_under_betting_accuracy_total_points,
    over_under_betting_accuracy_total_points_with_min_edge,
)

ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)
ou_scorer_edge_2 = OverUnderScorerTotalPointsMinEdge(
    line_col=BET365_LINE_COL,
    min_edge=2,
)
ou_scorer_edge_4 = OverUnderScorerTotalPointsMinEdge(
    line_col=BET365_LINE_COL,
    min_edge=4,
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_vals = cv_results[train_key]
        test_vals = cv_results[test_key]

        train_val = np.nanmean(train_vals)
        test_val = np.nanmean(test_vals)

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if "OU_Betting_Accuracy" in sc:
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
            n_valid = np.sum(~np.isnan(test_vals))
            print(f"  (valid folds: {n_valid}/{len(test_vals)})")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()


def summarize_walk_forward_total_points(
    predictions_df,
    daily_template,
    df_test_final,
    *,
    line_col=BET365_LINE_COL,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    line_lookup = (
        df_test_final.reset_index(drop=True)[[line_col]]
        .reset_index()
        .rename(columns={"index": "row_in_test_final"})
    )

    scored_predictions = predictions_df.merge(
        line_lookup,
        on="row_in_test_final",
        how="left",
        validate="many_to_one",
    ).copy()

    scored_predictions["date"] = pd.to_datetime(
        scored_predictions["date"],
        errors="coerce",
    ).dt.normalize()
    scored_predictions["y_true"] = pd.to_numeric(
        scored_predictions["y_true"],
        errors="coerce",
    )
    scored_predictions["y_pred"] = pd.to_numeric(
        scored_predictions["y_pred"],
        errors="coerce",
    )
    scored_predictions[line_col] = pd.to_numeric(
        scored_predictions[line_col],
        errors="coerce",
    )

    daily_context = daily_template.copy()
    daily_context["date"] = pd.to_datetime(
        daily_context["date"],
        errors="coerce",
    ).dt.normalize()
    daily_context = daily_context.drop(columns=["_walk_mae"], errors="ignore")

    daily_rows = []
    for current_day, day_df in scored_predictions.groupby("date", sort=True):
        context_row = daily_context.loc[daily_context["date"] == current_day].iloc[0].to_dict()
        context_row[metric_name] = over_under_betting_accuracy_total_points(
            y_true=day_df["y_true"].to_numpy(dtype=float),
            y_pred=day_df["y_pred"].to_numpy(dtype=float),
            betting_line=day_df[line_col].to_numpy(dtype=float),
        )
        daily_rows.append(context_row)

    daily_results = pd.DataFrame(daily_rows)

    y_true = scored_predictions["y_true"].to_numpy(dtype=float)
    y_pred = scored_predictions["y_pred"].to_numpy(dtype=float)
    betting_line = scored_predictions[line_col].to_numpy(dtype=float)
    pred_edge = y_pred - betting_line
    margin = np.abs(pred_edge)
    n_total = len(scored_predictions)

    threshold_rows = []
    for threshold in thresholds:
        mask = margin > threshold
        n_games = int(mask.sum())
        ou_acc = (
            np.nan
            if n_games == 0
            else over_under_betting_accuracy_total_points(
                y_true=y_true[mask],
                y_pred=y_pred[mask],
                betting_line=betting_line[mask],
            )
        )
        threshold_rows.append(
            {
                "threshold_abs_pred_edge_gt": threshold,
                "n_games": n_games,
                "pct_of_test": (n_games / n_total) if n_total else np.nan,
                "ou_betting_accuracy": ou_acc,
            }
        )

    threshold_results = pd.DataFrame(threshold_rows)
    return scored_predictions, daily_results, threshold_results


def run_day_by_day_walk_forward_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    raw_result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: mean_absolute_error(y_true, y_pred),
        target_col=TARGET_COL,
        max_games=max_games,
        metric_name="_walk_mae",
    )

    scored_predictions, daily_results, threshold_results = summarize_walk_forward_total_points(
        predictions_df=raw_result.predictions,
        daily_template=raw_result.daily_results,
        df_test_final=df_test_final,
        line_col=BET365_LINE_COL,
        metric_name=metric_name,
        thresholds=thresholds,
    )

    mean_metric = float(daily_results[metric_name].mean())
    print(f"{label} mean day-by-day {metric_name}: {mean_metric:.2%}")
    display(daily_results.style.format({metric_name: "{:.2%}"}))
    print(f"{label} thresholded walk-forward accuracy")
    display(
        threshold_results.style.format(
            {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
        )
    )
    return raw_result, scored_predictions, daily_results, threshold_results


In [16]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=50,
    step_games_between_tests=30,
    train_games=TRAIN_GAMES,
    min_train_games=int(TRAIN_GAMES * 0.5),
    max_folds=12,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           4391            51       2021-10-25     2025-03-17      2025-03-18    2025-03-24         2024
    2           4478            53       2021-10-25     2025-03-29      2025-03-30    2025-04-05         2024
    3           4564            51       2021-10-25     2025-04-09      2025-04-10    2025-04-25         2024
    4           4677            50       2021-10-25     2025-06-22      2025-10-27    2025-11-06         2025
    5           4762            51       2021-10-25     2025-11-10      2025-11-11    2025-11-17         2025
    6           4848            57       2021-10-25     2025-11-22      2025-11-23    2025-11-30         2025
    7           4946            61       2021-10-25     2025-12-05      2025-12-06    2025-12-18         2025
    8           5049            53       2021-10-25     2025-12-23      2025

In [17]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MAE: 15.54249
Validation MAE: 15.38317

Train RMSE: 19.48574
Validation RMSE: 19.13695

Train R2: 0.00000
Validation R2: -0.09323

Train OU_Betting_Accuracy: 49.86%
Validation OU_Betting_Accuracy: 50.00%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 49.55%
Validation OU_Betting_Accuracy_Edge_2: 49.83%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 50.31%
Validation OU_Betting_Accuracy_Edge_4: 50.60%
  (valid folds: 12/12)



In [18]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MAE: 10.88022
Validation MAE: 25.82996

Train RMSE: 13.83142
Validation RMSE: 41.30189

Train R2: 0.49600
Validation R2: -9.26018

Train OU_Betting_Accuracy: 69.48%
Validation OU_Betting_Accuracy: 49.29%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 73.12%
Validation OU_Betting_Accuracy_Edge_2: 50.75%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 76.64%
Validation OU_Betting_Accuracy_Edge_4: 50.75%
  (valid folds: 12/12)



In [19]:
xgb_reg_no_weights = XGBRegressor(
    max_depth=4,
    learning_rate=0.057,
    n_estimators=75,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-1,
    random_state=16,
)

cv_results_no_weights = cross_validate(
    xgb_reg_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost no sample weights")
print_metrics(cv_results_no_weights)


XGBoost no sample weights
Train MAE: 11.30628
Validation MAE: 14.15243

Train RMSE: 14.31707
Validation RMSE: 17.43831

Train R2: 0.46000
Validation R2: 0.09200

Train OU_Betting_Accuracy: 76.59%
Validation OU_Betting_Accuracy: 50.15%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 87.96%
Validation OU_Betting_Accuracy_Edge_2: 51.18%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 95.06%
Validation OU_Betting_Accuracy_Edge_4: 53.41%
  (valid folds: 12/12)



In [20]:
xgb_reg_no_weights.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg_no_weights.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")


Final test metrics
MSE: 317.27158
RMSE: 17.81212
MAE: 13.88581
OU_Betting_Accuracy: 50.85%
OU_Betting_Accuracy_Edge_2: 57.26%
OU_Betting_Accuracy_Edge_4: 48.39%


In [21]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg_no_weights,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)


def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBRegressor(**xgb_reg_no_weights.get_params())

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_no_weights, day_by_day_no_weights_predictions, day_by_day_no_weights_daily, day_by_day_no_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost no sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,298,100.0%,50.85%
1,1,202,67.8%,51.52%
2,2,120,40.3%,57.26%
3,3,65,21.8%,51.56%
4,4,31,10.4%,48.39%
5,5,16,5.4%,56.25%
6,6,9,3.0%,55.56%
7,7,4,1.3%,75.00%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


XGBoost no sample weights mean day-by-day OU_Betting_Accuracy: 48.49%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-03-01 00:00:00,5100,11,2021-12-23 00:00:00,2026-02-28 00:00:00,45.45%
1,2026-03-02 00:00:00,5100,4,2021-12-26 00:00:00,2026-03-01 00:00:00,75.00%
2,2026-03-03 00:00:00,5100,10,2021-12-26 00:00:00,2026-03-02 00:00:00,70.00%
3,2026-03-04 00:00:00,5100,6,2021-12-28 00:00:00,2026-03-03 00:00:00,66.67%
4,2026-03-05 00:00:00,5100,9,2021-12-28 00:00:00,2026-03-04 00:00:00,66.67%
5,2026-03-06 00:00:00,5100,7,2021-12-29 00:00:00,2026-03-05 00:00:00,14.29%
6,2026-03-07 00:00:00,5100,6,2021-12-31 00:00:00,2026-03-06 00:00:00,50.00%
7,2026-03-08 00:00:00,5100,10,2021-12-31 00:00:00,2026-03-07 00:00:00,50.00%
8,2026-03-09 00:00:00,5100,5,2022-01-02 00:00:00,2026-03-08 00:00:00,40.00%
9,2026-03-10 00:00:00,5100,11,2022-01-03 00:00:00,2026-03-09 00:00:00,54.55%


XGBoost no sample weights thresholded walk-forward accuracy


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,1,216,72.5%,50.00%
1,2,127,42.6%,47.20%
2,3,64,21.5%,52.38%


In [22]:
xgb_reg_weights = XGBRegressor(
    max_depth=4,
    learning_rate=0.057,
    n_estimators=75,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-1,
    random_state=16,
)

weighted_xgb = TemporalDecaySampleWeightRegressor(
    estimator=xgb_reg_weights,
    dates=df_dev["GAME_DATE"],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost with sample weights (per-fold decay)")
print_metrics(cv_results_weights)


XGBoost with sample weights (per-fold decay)
Train MAE: 12.50146
Validation MAE: 14.05239

Train RMSE: 16.00011
Validation RMSE: 17.52403

Train R2: 0.32537
Validation R2: 0.08510

Train OU_Betting_Accuracy: 60.72%
Validation OU_Betting_Accuracy: 52.45%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 65.39%
Validation OU_Betting_Accuracy_Edge_2: 52.10%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 71.08%
Validation OU_Betting_Accuracy_Edge_4: 54.63%
  (valid folds: 12/12)



In [23]:
weighted_xgb.fit(X_dev, y_dev)

y_pred_test_total = weighted_xgb.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")


Final test metrics
MSE: 327.80066
RMSE: 18.10527
MAE: 14.10888
OU_Betting_Accuracy: 49.83%
OU_Betting_Accuracy_Edge_2: 50.00%
OU_Betting_Accuracy_Edge_4: 52.34%


In [24]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=weighted_xgb,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)


def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBRegressor(**xgb_reg_weights.get_params())
    model = TemporalDecaySampleWeightRegressor(
        estimator=base_model,
        dates=train_df["GAME_DATE"],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_weights, day_by_day_weights_predictions, day_by_day_weights_daily, day_by_day_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost with sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,298,100.0%,49.83%
1,1,251,84.2%,49.19%
2,2,197,66.1%,50.00%
3,3,150,50.3%,52.35%
4,4,108,36.2%,52.34%
5,5,79,26.5%,56.41%
6,6,44,14.8%,50.00%
7,7,31,10.4%,45.16%
8,8,16,5.4%,37.50%
9,9,6,2.0%,16.67%


XGBoost with sample weights mean day-by-day OU_Betting_Accuracy: 49.95%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-03-01 00:00:00,5100,11,2021-12-23 00:00:00,2026-02-28 00:00:00,45.45%
1,2026-03-02 00:00:00,5100,4,2021-12-26 00:00:00,2026-03-01 00:00:00,25.00%
2,2026-03-03 00:00:00,5100,10,2021-12-26 00:00:00,2026-03-02 00:00:00,70.00%
3,2026-03-04 00:00:00,5100,6,2021-12-28 00:00:00,2026-03-03 00:00:00,33.33%
4,2026-03-05 00:00:00,5100,9,2021-12-28 00:00:00,2026-03-04 00:00:00,66.67%
5,2026-03-06 00:00:00,5100,7,2021-12-29 00:00:00,2026-03-05 00:00:00,42.86%
6,2026-03-07 00:00:00,5100,6,2021-12-31 00:00:00,2026-03-06 00:00:00,83.33%
7,2026-03-08 00:00:00,5100,10,2021-12-31 00:00:00,2026-03-07 00:00:00,37.50%
8,2026-03-09 00:00:00,5100,5,2022-01-02 00:00:00,2026-03-08 00:00:00,80.00%
9,2026-03-10 00:00:00,5100,11,2022-01-03 00:00:00,2026-03-09 00:00:00,54.55%


XGBoost with sample weights thresholded walk-forward accuracy


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,1,240,80.5%,50.21%
1,2,185,62.1%,50.54%
2,3,145,48.7%,47.92%


# OPTUNA

In [25]:
from nba_ou.modeling.optuna_total_points import (
    fit_best_xgb_total_points,
    select_best_trial_lexicographic,
    summarize_lexicographic_candidates,
    summarize_optuna_trials,
    tune_xgb_total_points_optuna,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev["GAME_DATE"],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=4.5 * 3600,
    objective_name="reg:squarederror",
    study_name="xgb_total_points_mae",
)

best_trial_lexi = select_best_trial_lexicographic(
    study,
    mae_tolerance_abs=0.05,
)

print("Optuna best by MAE only")
print("Trial:", study.best_trial.number)
print("Best CV MAE:", study.best_value)
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", study.best_trial.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", study.best_trial.user_attrs.get("mean_ou_acc_edge_4"))
print("Sample weight lambda:", study.best_trial.user_attrs.get("sample_weight_lambda"))

print()
print("Selected trial after MAE-first / OU-second ranking")
print("Trial:", best_trial_lexi.number)
print("CV MAE:", best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value))
print("Mean RMSE:", best_trial_lexi.user_attrs.get("mean_rmse"))
print("Mean R2:", best_trial_lexi.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", best_trial_lexi.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_4"))
print("Median best_iteration:", best_trial_lexi.user_attrs.get("median_best_iteration"))
print("Sample weight lambda:", best_trial_lexi.params.get("sample_weight_lambda"))
print("Params:")
for k, v in best_trial_lexi.params.items():
    print(f"{k}: {v}")

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_3": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

candidates_df = summarize_lexicographic_candidates(
    study,
    mae_tolerance_abs=0.05,
)

display(
    candidates_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_3": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)


[I 2026-04-11 02:39:38,865] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-04-11 02:48:15,222] Trial 0 finished with value: 13.744460735087403 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.0001422437941448231}. Best is trial 0 with value: 13.744460735087403.
[I 2026-04-11 03:02:05,699] Trial 1 finished with value: 13.730252125605823 and parameters: {'max_depth': 4, 'min_child_weight': 20.290108931287893, 'gamma': 0.3261777842309625, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.4213034780854249, 'learning_rate': 0.012620826760486504, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.0010239553604139541}. Best is trial 1 with value: 13.730252125605823.
[I 2026-04-11 03:07:54,538] Trial 2 finished with value: 13.739563979159739 and parameters: {'

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,13,13.5916,17.0730,0.1301,56.45%,59.20%,64.71%,66.90%,143,112,4,40.091466,1.377688,0.846384,0.795010,0.030599,0.355407,3.298533,0.002240
1,29,13.6007,17.0239,0.1351,55.36%,61.25%,64.00%,61.11%,107,108,4,17.539262,0.259163,0.803883,0.360131,0.047525,2.393395,36.339572,0.000157
2,26,13.6179,17.0821,0.1293,58.03%,59.11%,60.50%,66.47%,103,54,4,27.250316,0.110730,0.772781,0.770338,0.058818,5.777663,32.402186,0.003221
3,6,13.6216,17.0290,0.1345,57.56%,61.49%,59.44%,57.79%,136,90,4,21.354130,1.318300,0.833701,0.787420,0.040420,0.264001,42.379142,0.000127
4,25,13.6223,17.0864,0.1295,59.32%,59.21%,60.00%,63.18%,101,109,4,27.233923,1.183759,0.773526,0.770267,0.043712,7.863742,24.356740,0.001555
5,19,13.6264,17.0699,0.1315,54.69%,60.55%,62.50%,69.23%,108,78,3,34.611933,1.411200,0.724319,0.682733,0.035063,1.242166,7.277097,0.001723
6,20,13.6337,17.0694,0.1318,54.80%,58.35%,60.23%,56.78%,89,50,3,59.589827,0.687202,0.884765,0.745642,0.045666,0.032633,2.308014,0.003415
7,22,13.6350,17.0476,0.1332,57.11%,59.29%,61.47%,53.12%,132,95,4,21.827024,1.038456,0.723720,0.760552,0.030533,1.105641,4.026092,0.001429
8,9,13.6383,17.0912,0.1289,54.70%,58.69%,61.31%,59.68%,85,68,4,15.509831,1.261227,0.669576,0.722762,0.049743,3.380961,1.306366,0.002083
9,23,13.6437,17.0343,0.1349,55.10%,60.55%,63.18%,46.37%,163,96,3,44.797418,1.770217,0.821463,0.707665,0.034611,0.309567,8.329379,0.000623


,trial,value_mae,mean_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,mae_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,25,13.6223,13.6223,17.0864,0.1295,59.32%,59.21%,60.00%,63.18%,101,109,13.641605,4,27.233923,1.183759,0.773526,0.770267,0.043712,7.863742,24.356740,0.001555
1,26,13.6179,13.6179,17.0821,0.1293,58.03%,59.11%,60.50%,66.47%,103,54,13.641605,4,27.250316,0.110730,0.772781,0.770338,0.058818,5.777663,32.402186,0.003221
2,6,13.6216,13.6216,17.0290,0.1345,57.56%,61.49%,59.44%,57.79%,136,90,13.641605,4,21.354130,1.318300,0.833701,0.787420,0.040420,0.264001,42.379142,0.000127
3,22,13.6350,13.6350,17.0476,0.1332,57.11%,59.29%,61.47%,53.12%,132,95,13.641605,4,21.827024,1.038456,0.723720,0.760552,0.030533,1.105641,4.026092,0.001429
4,13,13.5916,13.5916,17.0730,0.1301,56.45%,59.20%,64.71%,66.90%,143,112,13.641605,4,40.091466,1.377688,0.846384,0.795010,0.030599,0.355407,3.298533,0.002240
5,29,13.6007,13.6007,17.0239,0.1351,55.36%,61.25%,64.00%,61.11%,107,108,13.641605,4,17.539262,0.259163,0.803883,0.360131,0.047525,2.393395,36.339572,0.000157
6,20,13.6337,13.6337,17.0694,0.1318,54.80%,58.35%,60.23%,56.78%,89,50,13.641605,3,59.589827,0.687202,0.884765,0.745642,0.045666,0.032633,2.308014,0.003415
7,9,13.6383,13.6383,17.0912,0.1289,54.70%,58.69%,61.31%,59.68%,85,68,13.641605,4,15.509831,1.261227,0.669576,0.722762,0.049743,3.380961,1.306366,0.002083
8,19,13.6264,13.6264,17.0699,0.1315,54.69%,60.55%,62.50%,69.23%,108,78,13.641605,3,34.611933,1.411200,0.724319,0.682733,0.035063,1.242166,7.277097,0.001723


In [26]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model = fit_best_xgb_total_points(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df["GAME_DATE"],
        sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
        trial=best_trial_lexi,
        objective_name="reg:squarederror",
    )
    return model.predict(X_test)


day_by_day_optuna, day_by_day_optuna_predictions, day_by_day_optuna_daily, day_by_day_optuna_thresholds = run_day_by_day_walk_forward_evaluation(
    label="Optuna-selected XGBoost",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)

total_df = df_dev.tail(TRAIN_GAMES)


Optuna-selected XGBoost mean day-by-day OU_Betting_Accuracy: 51.75%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-03-01 00:00:00,5100,11,2021-12-23 00:00:00,2026-02-28 00:00:00,45.45%
1,2026-03-02 00:00:00,5100,4,2021-12-26 00:00:00,2026-03-01 00:00:00,25.00%
2,2026-03-03 00:00:00,5100,10,2021-12-26 00:00:00,2026-03-02 00:00:00,80.00%
3,2026-03-04 00:00:00,5100,6,2021-12-28 00:00:00,2026-03-03 00:00:00,50.00%
4,2026-03-05 00:00:00,5100,9,2021-12-28 00:00:00,2026-03-04 00:00:00,66.67%
5,2026-03-06 00:00:00,5100,7,2021-12-29 00:00:00,2026-03-05 00:00:00,28.57%
6,2026-03-07 00:00:00,5100,6,2021-12-31 00:00:00,2026-03-06 00:00:00,83.33%
7,2026-03-08 00:00:00,5100,10,2021-12-31 00:00:00,2026-03-07 00:00:00,62.50%
8,2026-03-09 00:00:00,5100,5,2022-01-02 00:00:00,2026-03-08 00:00:00,20.00%
9,2026-03-10 00:00:00,5100,11,2022-01-03 00:00:00,2026-03-09 00:00:00,45.45%


Optuna-selected XGBoost thresholded walk-forward accuracy


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,1,213,71.5%,50.72%
1,2,149,50.0%,49.66%
2,3,81,27.2%,50.63%


In [27]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")
sample_weight_dates_dev = total_df["GAME_DATE"]

best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=sample_weight_dates_dev,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=best_model,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

df_to_train_split_rows = df_to_train.copy().tail(TRAIN_GAMES)

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")
sample_weight_dates_full = df_to_train_split_rows["GAME_DATE"]

production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"three_seasons_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="three_seasons_total_points",
        prediction_source="three_seasons_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get("mean_best_iteration"),
        median_best_iteration=best_trial_lexi.user_attrs.get("median_best_iteration"),
        cv_mae=float(best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get("mean_rmse"),
        cv_ou_acc=best_trial_lexi.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
        train_games=TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons_weighted/",
    metadata=metadata,
)

print(
    f"Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration."
)
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Final test metrics
MSE: 325.32840
RMSE: 18.03686
MAE: 14.07795
OU_Betting_Accuracy: 51.19%
OU_Betting_Accuracy_Edge_2: 51.28%
OU_Betting_Accuracy_Edge_4: 43.14%


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,298,100.0%,51.19%
1,1,218,73.2%,52.34%
2,2,160,53.7%,51.28%
3,3,89,29.9%,44.32%
4,4,51,17.1%,43.14%
5,5,23,7.7%,34.78%
6,6,12,4.0%,41.67%
7,7,5,1.7%,20.00%
8,8,4,1.3%,25.00%
9,9,2,0.7%,50.00%


Production model trained on 5100 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons_weighted/three_seasons_xgb_total_points_08_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons_weighted/three_seasons_xgb_total_points_08_04_26.meta.json
